# DMS 行为识别算法 A · Colab 一键训练

在免费 T4 GPU 上训练 8 类危险驾驶行为检测器（约 20-40 分钟）。

类别：`hand_on_wheel / phone_use / calling / drinking / reach_behind / cigarette / no_seatbelt / seatbelt`

## 步骤
1. 打开 Colab → 运行时 → 更改运行时类型 → **GPU (T4)**
2. 顺序运行下面每个 cell
3. 训完后下载 `unified.pt` 放到本机 `models/` 目录

## 1. 环境 & Roboflow 下载

In [ ]:
!pip install -q ultralytics roboflow pyyaml
import os
# 把 API Key 放环境变量，不直接写代码里
os.environ['ROBOFLOW_API_KEY'] = 'PASTE_YOUR_KEY_HERE'  # ← 填你的 Roboflow Private API Key

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])

print('[1/3] distracted_driving ...')
rf.workspace('yolov8-ei4l6').project('distracted-driving-yolov8').version(6)\
    .download('yolov8', location='data/distracted_driving')
print('[2/3] cigarette ...')
rf.workspace('yolov8-jymgm').project('cigarette-wkkgi').version(5)\
    .download('yolov8', location='data/cigarette')
print('[3/3] seatbelt ...')
rf.workspace('seatbelttraining-7yh0f').project('seatbelt-detection-lb1ec').version(4)\
    .download('yolov8', location='data/seatbelt')

## 2. 合并 → 8 类统一数据集

In [ ]:
import shutil
from pathlib import Path
import yaml

SRC = Path('data')
DST = Path('data/dms_unified')

UNIFIED = ['hand_on_wheel','phone_use','calling','drinking',
           'reach_behind','cigarette','no_seatbelt','seatbelt']
MAP = {
    'distracted_driving': {0:0, 1:1, 2:2, 3:3, 4:4},
    'cigarette':          {0:5},
    'seatbelt':           {0:6, 1:7},
}

def remap(src, dst, m):
    out = []
    with open(src) as f:
        for line in f:
            p = line.strip().split()
            if not p: continue
            oid = int(p[0])
            if oid in m: out.append(f'{m[oid]} {p[1]} {p[2]} {p[3]} {p[4]}')
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text('\n'.join(out))

if DST.exists(): shutil.rmtree(DST)
for sub, m in MAP.items():
    for split in ['train','valid']:
        img_src = SRC/sub/split/'images'
        lbl_src = SRC/sub/split/'labels'
        if not img_src.exists(): continue
        for img in img_src.iterdir():
            new = f'{sub}_{img.name}'
            (DST/split/'images').mkdir(parents=True, exist_ok=True)
            shutil.copy(img, DST/split/'images'/new)
            lbl = lbl_src / (img.stem + '.txt')
            if lbl.exists():
                remap(lbl, DST/split/'labels'/f'{sub}_{img.stem}.txt', m)

(DST/'data.yaml').write_text(yaml.dump({
    'path': str(DST.resolve()),
    'train': 'train/images', 'val': 'valid/images',
    'nc': 8, 'names': UNIFIED,
}, sort_keys=False, allow_unicode=True))

print('[ok] train:', len(list((DST/'train/images').iterdir())),
      'val:', len(list((DST/'valid/images').iterdir())))

## 3. 训练（T4 GPU 约 25-40 分钟）

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(
    data='data/dms_unified/data.yaml',
    epochs=40, imgsz=640, batch=32,
    device=0, project='runs', name='unified',
    patience=15, optimizer='AdamW', lr0=0.001,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    translate=0.1, scale=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1,
    box=7.5, cls=0.5, dfl=1.5,
)
print('done. best ->', 'runs/unified/weights/best.pt')

## 4. 下载 unified.pt 回本机

In [ ]:
from google.colab import files
files.download('runs/unified/weights/best.pt')
# 本机收到后重命名为 unified.pt 放入 models/ 目录

## 5. 本机使用

```bash
# 1. 把下载的 best.pt 放到 behavior_algo_a/models/unified.pt
# 2. 运行
python live_client.py --unified models/unified.pt --style monitor
```